# Grids for New Projections (Arctic & Antarctic)

Standard lat/lon grids degrade at high latitudes — cell aspect ratios become extreme,
and the MOM6 timestep constraint can become prohibitively tight near the poles.
For Arctic or Antarctic domains, `mom6_forge` can build a grid in a **projected
coordinate system** (e.g. polar stereographic) and reproject it to the geographic
coordinates MOM6 expects.

Two approaches are shown here:
1. **Interactive** — use the `GridCreator` widget and select *From Projection* mode.
2. **Programmatic** — call `Grid.from_projection()` directly.

## Approach 1: Interactive GridCreator widget

Launch the widget, choose **From Projection** in the method dropdown, select
**Arctic Polar Stereographic (EPSG:3995)** or **Antarctic Polar Stereographic (EPSG:3031)**
from the CRS dropdown, set a resolution, then drag a rectangle on the polar map to
define the domain. The widget calls `Grid.from_projection()` under the hood and saves
the result to `GridLibrary/`.

In [ ]:
from mom6_forge.grid_creator import GridCreator

gc = GridCreator()
gc

## Approach 2: Programmatic — `Grid.from_projection()`

Pass a pyproj CRS (as an EPSG string or integer) along with the domain extents
in **metres** in the native projection. The method reprojects to geographic
coordinates and computes grid metrics using exact great-circle geometry.

| CRS | Region |
|---|---|
| `"EPSG:3995"` | Arctic Polar Stereographic |
| `"EPSG:3031"` | Antarctic Polar Stereographic |

### Arctic domain (EPSG:3995)

EPSG:3995 is centred at the North Pole. Extents are in metres from the pole.
The example below creates a 2000 km × 2000 km domain at 50 km resolution.

In [ ]:
from mom6_forge.grid import Grid

arctic_grid = Grid.from_projection(
    crs="EPSG:3995",
    x_min=-1_000_000,  # metres from pole
    x_max= 1_000_000,
    y_min=-1_000_000,
    y_max= 1_000_000,
    resolution_m=50_000,  # 50 km
    name="arctic_50km",
)

print(f"Grid size: {arctic_grid.nx} x {arctic_grid.ny}")
print(f"Lat range: {arctic_grid.tlat.values.min():.1f}° to {arctic_grid.tlat.values.max():.1f}°")

In [ ]:
import cartopy.crs as ccrs
import matplotlib.pyplot as plt

fig, ax = plt.subplots(subplot_kw={"projection": ccrs.NorthPolarStereo()}, figsize=(6, 6))
ax.set_extent([-180, 180, 45, 90], crs=ccrs.PlateCarree())
ax.coastlines()
ax.gridlines()
ax.pcolormesh(
    arctic_grid.tlon, arctic_grid.tlat,
    arctic_grid.tlat,   # colour by latitude as a sanity check
    transform=ccrs.PlateCarree(), cmap="Blues", alpha=0.6,
)
ax.set_title("Arctic domain — EPSG:3995")
plt.tight_layout()

### Antarctic domain (EPSG:3031)

EPSG:3031 is centred at the South Pole. The example below creates a
3000 km × 3000 km domain (covering most of Antarctica) at 25 km resolution.

In [ ]:
antarctic_grid = Grid.from_projection(
    crs="EPSG:3031",
    x_min=-1_500_000,
    x_max= 1_500_000,
    y_min=-1_500_000,
    y_max= 1_500_000,
    resolution_m=25_000,  # 25 km
    name="antarctic_25km",
)

print(f"Grid size: {antarctic_grid.nx} x {antarctic_grid.ny}")
print(f"Lat range: {antarctic_grid.tlat.values.min():.1f}° to {antarctic_grid.tlat.values.max():.1f}°")

In [ ]:
fig, ax = plt.subplots(subplot_kw={"projection": ccrs.SouthPolarStereo()}, figsize=(6, 6))
ax.set_extent([-180, 180, -90, -45], crs=ccrs.PlateCarree())
ax.coastlines()
ax.gridlines()
ax.pcolormesh(
    antarctic_grid.tlon, antarctic_grid.tlat,
    antarctic_grid.tlat,
    transform=ccrs.PlateCarree(), cmap="Blues_r", alpha=0.6,
)
ax.set_title("Antarctic domain — EPSG:3031")
plt.tight_layout()

## Next steps

Once you have a polar grid, pass it to `Topo` and then to `Case` exactly as you
would any other `Grid` object — the rest of the CrocoDash workflow is unchanged.
See [Getting Started](../tutorials/crocodash_tutorial.ipynb) for the full case
setup walkthrough.